RFM Feature Engineering

In [2]:
# STEP 1: Reduced Dataset Load 

import pandas as pd

df = pd.read_csv("retail_reduced_customers.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [3]:
#df.head()

In [4]:
#df.info()

In [5]:
#STEP 2: Reference Date to calculate Recency
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
reference_date

Timestamp('2011-12-10 12:50:00')

In [6]:
#STEP 3: RFM Table Creation

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                    # Frequency
    'TotalAmount': 'sum'                                      # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12350.0,310,1,334.40
12352.0,36,8,2506.04


In [7]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 12346.0 to 18287.0
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Recency    3000 non-null   int64  
 1   Frequency  3000 non-null   int64  
 2   Monetary   3000 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 93.8 KB


In [8]:
#STEP 4: Basic Sanity Check
rfm.describe()

,Recency,Frequency,Monetary
count,3000.000000,3000.000000,3000.000000
mean,91.494333,4.244333,2058.208624
std,99.817803,7.228835,8818.620553
min,1.000000,1.000000,3.750000
25%,18.000000,1.000000,307.702500
50%,50.000000,2.000000,678.965000
75%,137.250000,5.000000,1705.992500
max,374.000000,201.000000,259657.300000


NEXT STEP: RFM SCORING & CUSTOMER SEGMENTATION

In [9]:
# STEP 1: Recency Score (R)

rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5, 4, 3, 2, 1]
)


In [10]:
# STEP 2: Frequency Score (F)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

In [11]:
# STEP 3: Monetary Score (M)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
)


In [12]:
# STEP 4: Check Scores
rfm[['Recency','R_Score','Frequency','F_Score','Monetary','M_Score']].head()


,Recency,R_Score,Frequency,F_Score,Monetary,M_Score
CustomerID,,,,,,
12346.0,326,1,1,1,77183.60,5
12347.0,2,5,7,5,4310.00,5
12348.0,75,2,4,4,1797.24,4
12350.0,310,1,1,1,334.40,2
12352.0,36,3,8,5,2506.04,5


In [13]:
#STEP 5: RFM Score Combine

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

In [14]:
rfm['RFM_Score']

CustomerID
12346.0    115
12347.0    555
12348.0    244
12350.0    112
12352.0    355
          ... 
18274.0    421
18276.0    322
18278.0    221
18282.0    531
18287.0    344
Name: RFM_Score, Length: 3000, dtype: object

In [15]:
# STEP 6: Simple Segmentation (optional BUT useful)

def segment(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4:
        return 'Champion'
    elif row['F_Score'] >= 4:
        return 'Loyal'
    elif row['R_Score'] >= 4:
        return 'Potential'
    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'At Risk'
    else:
        return 'Need Attention'

rfm['Segment'] = rfm.apply(segment, axis=1)

In [16]:
rfm['Segment']

CustomerID
12346.0           At Risk
12347.0          Champion
12348.0             Loyal
12350.0           At Risk
12352.0             Loyal
                ...      
18274.0         Potential
18276.0    Need Attention
18278.0           At Risk
18282.0         Potential
18287.0             Loyal
Name: Segment, Length: 3000, dtype: object

In [28]:
#RFM Dataset Save ा
#rfm.to_csv("retail_rfm_customers.csv", index=True)  # CustomerID index म्हणून राहील

In [29]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 12346.0 to 18287.0
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Recency          3000 non-null   int64   
 1   Frequency        3000 non-null   int64   
 2   Monetary         3000 non-null   float64 
 3   R_Score          3000 non-null   category
 4   F_Score          3000 non-null   category
 5   M_Score          3000 non-null   category
 6   RFM_Score        3000 non-null   object  
 7   Segment          3000 non-null   object  
 8   Churn            3000 non-null   int64   
 9   Business_Action  3000 non-null   object  
dtypes: category(3), float64(1), int64(3), object(3)
memory usage: 196.9+ KB


In [30]:
rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Churn,Business_Action
CustomerID,,,,,,,,,,
12346.0,326,1,77183.60,1,1,5,115,At Risk,1,Win-back offers
12347.0,2,7,4310.00,5,5,5,555,Champion,0,VIP treatment
12348.0,75,4,1797.24,2,4,4,244,Loyal,0,Retention rewards
12350.0,310,1,334.40,1,1,2,112,At Risk,1,Win-back offers
12352.0,36,8,2506.04,3,5,5,355,Loyal,0,Retention rewards


NEXT STEP: CHURN LABEL CREATION 

In [31]:
# Step 1: Churn column create 
rfm['Churn'] = rfm['Recency'].apply(
    lambda x: 1 if x > 90 else 0
)

In [32]:
# Step 2: Verify 
rfm[['Recency', 'Churn']].head(5)

,Recency,Churn
CustomerID,,
12346.0,326,1
12347.0,2,0
12348.0,75,0
12350.0,310,1
12352.0,36,0


In [33]:
# Step 3: Overall churn rate 
rfm['Churn'].value_counts(normalize=True) * 100

Churn
0    67.1
1    32.9
Name: proportion, dtype: float64

In [34]:
# STEP 4: Churn vs Segment (Insight)
pd.crosstab(rfm['Segment'], rfm['Churn'])

Churn,0,1
Segment,,
At Risk,101,645
Champion,777,0
Loyal,282,141
Need Attention,415,201
Potential,438,0


NEXT STEP: Business Action column add

In [35]:
# Option 1: Dictionary mapping (BEST PRACTICE)
action_map = {
    'At Risk': 'Win-back offers',
    'Loyal': 'Retention rewards',
    'Need Attention': 'Discounts',
    'Potential': 'Upsell',
    'Champion': 'VIP treatment'
}

rfm['Business_Action'] = rfm['Segment'].map(action_map)



In [36]:
# STEP 2: Verify
rfm[['Segment', 'Churn', 'Business_Action']].head(10)

,Segment,Churn,Business_Action
CustomerID,,,
12346.0,At Risk,1,Win-back offers
12347.0,Champion,0,VIP treatment
12348.0,Loyal,0,Retention rewards
12350.0,At Risk,1,Win-back offers
12352.0,Loyal,0,Retention rewards
12353.0,At Risk,1,Win-back offers
12354.0,At Risk,1,Win-back offers
12355.0,At Risk,1,Win-back offers
12356.0,Potential,0,Upsell


NEXT STEP: AOV & CLV Calculation (Step-by-Step)

In [37]:
# STEP 7.1: AOV (Average Order Value)
# AOV = Total Revenue / Total Orders

# Per-customer AOV
aov = (
    df.groupby('CustomerID')
      .agg(
          Total_Revenue=('TotalAmount', 'sum'),
          Total_Orders=('InvoiceNo', 'nunique')
      )
)

aov['AOV'] = aov['Total_Revenue'] / aov['Total_Orders']
aov.head()



,Total_Revenue,Total_Orders,AOV
CustomerID,,,
12346.0,77183.60,1,77183.600000
12347.0,4310.00,7,615.714286
12348.0,1797.24,4,449.310000
12350.0,334.40,1,334.400000
12352.0,2506.04,8,313.255000


In [38]:
# Check
aov['AOV'].describe()

count     3000.000000
mean       438.145330
std       2144.784646
min          3.450000
25%        178.845000
50%        292.285000
75%        432.542500
max      84236.250000
Name: AOV, dtype: float64

In [39]:
# STEP 7.2: AOV plus RFM  join

rfm = rfm.merge(aov[['AOV']], left_index=True, right_index=True, how='left')
rfm.head()


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Churn,Business_Action,AOV
CustomerID,,,,,,,,,,,
12346.0,326,1,77183.60,1,1,5,115,At Risk,1,Win-back offers,77183.600000
12347.0,2,7,4310.00,5,5,5,555,Champion,0,VIP treatment,615.714286
12348.0,75,4,1797.24,2,4,4,244,Loyal,0,Retention rewards,449.310000
12350.0,310,1,334.40,1,1,2,112,At Risk,1,Win-back offers,334.400000
12352.0,36,8,2506.04,3,5,5,355,Loyal,0,Retention rewards,313.255000


In [40]:
# STEP 7.3: CLV (Simple & Viva-Friendly)
# Logic :
# CLV = AOV × Purchase Frequency × Customer Lifespan
#Lifespan → 1 year (assumption)

customer_lifespan = 1  # year

rfm['CLV'] = rfm['AOV'] * rfm['Frequency'] * customer_lifespan
rfm[['AOV', 'Frequency', 'CLV']].head()



,AOV,Frequency,CLV
CustomerID,,,
12346.0,77183.600000,1,77183.60
12347.0,615.714286,7,4310.00
12348.0,449.310000,4,1797.24
12350.0,334.400000,1,334.40
12352.0,313.255000,8,2506.04


In [41]:
# STEP 7.4: CLV by Segment (Business Insight)

rfm.groupby('Segment')['CLV'].mean().sort_values(ascending=False)

Segment
Champion          5040.916229
Loyal             2107.300189
Potential         1039.283950
Need Attention     865.635341
At Risk            506.713461
Name: CLV, dtype: float64

In [42]:
# STEP 7.5: Revenue at Risk (Power Move)

revenue_at_risk = rfm.loc[rfm['Segment'] == 'At Risk', 'CLV'].sum()
revenue_at_risk

378008.242

In [43]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 12346.0 to 18287.0
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Recency          3000 non-null   int64   
 1   Frequency        3000 non-null   int64   
 2   Monetary         3000 non-null   float64 
 3   R_Score          3000 non-null   category
 4   F_Score          3000 non-null   category
 5   M_Score          3000 non-null   category
 6   RFM_Score        3000 non-null   object  
 7   Segment          3000 non-null   object  
 8   Churn            3000 non-null   int64   
 9   Business_Action  3000 non-null   object  
 10  AOV              3000 non-null   float64 
 11  CLV              3000 non-null   float64 
dtypes: category(3), float64(3), int64(3), object(3)
memory usage: 308.3+ KB


CHURN PREDICTION USING ML ALGORITHM -- LOGISTIC REGRESSION

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Split
X = rfm[['Recency', 'Frequency', 'Monetary', 'AOV', 'CLV']]
y = rfm['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


# Scaling but keeping column names
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

# Train model on DataFrame (with feature names)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

accuracy = model.score(X_test_scaled, y_test)
print("Accuracy:", accuracy)

Test the prediction model for new customer

In [67]:
# STEP 2: Data model 

import pandas as pd

new_customer = pd.DataFrame({
    'Recency': [120],
    'Frequency': [1],
    'Monetary': [500],
    'AOV': [500],
    'CLV': [500]
})



In [68]:
# STEP 3: Prediction 
prediction = model.predict(new_customer)
prediction


array([1], dtype=int64)

In [69]:
# STEP 4: Human-readable answer 
if prediction[0] == 1:
    print("Customer is likely to churn")
else:
    print("Customer is likely to stay")


Customer is likely to churn


In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270850 entries, 0 to 270849
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    270850 non-null  int64         
 1   StockCode    270850 non-null  object        
 2   Description  270850 non-null  object        
 3   Quantity     270850 non-null  int64         
 4   InvoiceDate  270850 non-null  datetime64[ns]
 5   UnitPrice    270850 non-null  float64       
 6   CustomerID   270850 non-null  float64       
 7   Country      270850 non-null  object        
 8   TotalAmount  270850 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 18.6+ MB


In [77]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 12346.0 to 18287.0
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Recency          3000 non-null   int64   
 1   Frequency        3000 non-null   int64   
 2   Monetary         3000 non-null   float64 
 3   R_Score          3000 non-null   category
 4   F_Score          3000 non-null   category
 5   M_Score          3000 non-null   category
 6   RFM_Score        3000 non-null   object  
 7   Segment          3000 non-null   object  
 8   Churn            3000 non-null   int64   
 9   Business_Action  3000 non-null   object  
 10  AOV              3000 non-null   float64 
 11  CLV              3000 non-null   float64 
dtypes: category(3), float64(3), int64(3), object(3)
memory usage: 308.3+ KB


TO MAKE POWERBI READY DATASET AND SAVE

In [78]:
#rfm_reset = rfm.reset_index()
#rfm_reset.rename(columns={'index': 'CustomerID'}, inplace=True)

#rfm_reset.to_csv("rfm_powerbi.csv", index=False)

In [79]:
#pdf=pd.read_csv("rfm_powerbi.csv")
#pdf

Save ML trained Model

In [80]:
import joblib

joblib.dump(model, "churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")


['scaler.pkl']